In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/lunecid/Documents/Project/DatoryLab/Data/부산광역시_대규모점포 등록현황_20230630.csv', encoding='cp949')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   광역자치단체        96 non-null     object 
 1   기초자치단체        96 non-null     object 
 2   상호            96 non-null     object 
 3   업태            96 non-null     object 
 4   도로명주소         96 non-null     object 
 5   위도            96 non-null     float64
 6   경도            96 non-null     float64
 7   총 매장면적(제곱미터)  96 non-null     int64  
 8   판매면적(제곱미터)    96 non-null     int64  
 9   용역제공면적(제곱미터)  63 non-null     float64
 10  실제 영업개시일      96 non-null     object 
dtypes: float64(3), int64(2), object(6)
memory usage: 8.4+ KB


,광역자치단체,기초자치단체,상호,업태,도로명주소,위도,경도,총 매장면적(제곱미터),판매면적(제곱미터),용역제공면적(제곱미터),실제 영업개시일
0,부산광역시,중구,롯데마트 광복점,대형마트,부산광역시 중구 중앙대로 2(중앙동7가),35.098309,129.036705,12347,9246,3101.0,2014.8.28
1,부산광역시,영도구,홈플러스 영도점,대형마트,부산광역시 영도구 대교로14번길 57(봉래동2가),35.095901,129.044231,5340,4860,481.0,2005.2.22
2,부산광역시,부산진구,홈플러스 서면점,대형마트,부산광역시 부산진구 동천로 4(전포동),35.149434,129.063978,11036,11036,NaN,1999.7.6
3,부산광역시,부산진구,홈플러스 가야점,대형마트,부산광역시 부산진구 가야대로 506(가야동),35.152464,129.027317,10063,9624,439.0,2002.8.21
4,부산광역시,부산진구,이마트 트레이더스서면점,대형마트,부산광역시 부산진구 시민공원로 31(부암동),35.163902,129.052554,11499,11499,NaN,2005.4.27


In [8]:
df['행정동'] = df['도로명주소'].str.extract(r'\(([^()\s]*동[^()]*)\)')

In [10]:
df['행정동']

0            중앙동7가
1            봉래동2가
2              전포동
3              가야동
4              부암동
          ...     
91             NaN
92             남산동
93             NaN
94    토성동1가, 경동 리인
95         중동, 엘시티
Name: 행정동, Length: 96, dtype: object

In [12]:
pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 17.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [15]:
# 2. GeoDataFrame으로 변환 (위도, 경도 → Point 객체)
df['geometry'] = df.apply(lambda row: Point(row['경도'], row['위도']), axis=1)
gdf_mart = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 3. SHP 파일 (부산 행정동) 불러오기
gdf_dong = gpd.read_file('/Users/lunecid/Downloads/hangjeongdong_pusan.geojson')  # 필요한 경우 EPSG 확인

# 4. CRS 맞추기 (위도경도와 동일하게 맞추기)
gdf_dong = gdf_dong.to_crs("EPSG:4326")

# 5. 공간 조인: 마트 위치가 어느 행정동에 속하는지
gdf_joined = gpd.sjoin(gdf_mart, gdf_dong, how="left", predicate='intersects')

In [31]:
gdf_joined 

,광역자치단체,기초자치단체,상호,업태,도로명주소,위도,경도,총 매장면적(제곱미터),판매면적(제곱미터),용역제공면적(제곱미터),...,OBJECTID,adm_nm,adm_cd,adm_cd2,sgg,sido,sidonm,sggnm,행정동코드,동
0,부산광역시,중구,롯데마트 광복점,대형마트,부산광역시 중구 중앙대로 2(중앙동7가),35.098309,129.036705,12347,9246,3101.0,...,425,부산광역시 중구 중앙동,2101051,2611051000,26110,26,부산광역시,중구,2101051,중앙동
1,부산광역시,영도구,홈플러스 영도점,대형마트,부산광역시 영도구 대교로14번길 57(봉래동2가),35.095901,129.044231,5340,4860,481.0,...,462,부산광역시 영도구 봉래1동,2104059,2620059000,26200,26,부산광역시,영도구,2104059,봉래1동
2,부산광역시,부산진구,홈플러스 서면점,대형마트,부산광역시 부산진구 동천로 4(전포동),35.149434,129.063978,11036,11036,NaN,...,487,부산광역시 부산진구 전포1동,2105083,2623060000,26230,26,부산광역시,부산진구,2105083,전포1동
3,부산광역시,부산진구,홈플러스 가야점,대형마트,부산광역시 부산진구 가야대로 506(가야동),35.152464,129.027317,10063,9624,439.0,...,480,부산광역시 부산진구 가야2동,2105072,2623072000,26230,26,부산광역시,부산진구,2105072,가야2동
4,부산광역시,부산진구,이마트 트레이더스서면점,대형마트,부산광역시 부산진구 시민공원로 31(부암동),35.163902,129.052554,11499,11499,NaN,...,476,부산광역시 부산진구 부암1동,2105064,2623064000,26230,26,부산광역시,부산진구,2105064,부암1동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,부산광역시,기장군,오시리아스퀘어,그 밖의 대규모점포,부산광역시 기장군 기장읍 기장해안로 98,35.187708,129.212342,12916,10638,2278.0,...,620,부산광역시 기장군 기장읍,2131011,2671025000,26710,26,부산광역시,기장군,2131011,NaN
92,부산광역시,금정구,남산(농수산)시장,그 밖의 대규모점포,부산광역시 금정구 금강로 700(남산동),35.269832,129.089293,4477,2931,1546.0,...,574,부산광역시 금정구 남산동,2111068,2641067000,26410,26,부산광역시,금정구,2111068,남산동
93,부산광역시,기장군,탑스퀘어,그 밖의 대규모점포,부산광역시 기장군 정관읍 정관중앙로 45,35.321195,129.178283,35300,10545,24755.0,...,622,부산광역시 기장군 정관읍,2131013,2671025600,26710,26,부산광역시,기장군,2131013,NaN
94,부산광역시,서구,경동타워 쇼핑,그 밖의 대규모점포,"부산광역시 서구 보수대로 27 (토성동1가, 경동 리인)",35.099271,129.024246,5026,3676,1350.0,...,443,부산광역시 서구 충무동,2102064,2614064000,26140,26,부산광역시,서구,2102064,충무동


In [41]:
gdf_joined.iloc[31]

광역자치단체                                    부산광역시
기초자치단체                                      기장군
상호                                     메가마트 기장점
업태                                         대형마트
도로명주소               부산광역시 기장군 일광면 기장대로 673(삼성리)
위도                                    35.254692
경도                                   129.221856
총 매장면적(제곱미터)                               7372
판매면적(제곱미터)                                 6620
용역제공면적(제곱미터)                              752.0
실제 영업개시일                              2010.6.15
행정동                                         NaN
geometry        POINT (129.2218562 35.25469204)
index_right                                 198
OBJECTID                                    623
adm_nm                            부산광역시 기장군 일광면
adm_cd                                  2131031
adm_cd2                              2671031000
sgg                                       26710
sido                                         26
sidonm                                  

In [42]:
gdf_joined['d1'] = gdf_joined['adm_nm'].str.extract(r'(\w+(동|읍|리|면))$')[0]
gdf_joined['행정동코드'] = gdf_joined['adm_cd']

In [43]:
print(gdf_joined['d1'].to_string())

0       중앙동
1      봉래1동
2      전포1동
3      가야2동
4      부암1동
5      부전1동
6      부암1동
7       명륜동
8      온천1동
9      온천2동
10     문현2동
11     감만2동
12     화명3동
13      금곡동
14      중1동
15      우3동
16      우2동
17     반여1동
18     신평2동
19     장림1동
20     장림2동
21     구서1동
22     연산2동
23     거제2동
24     연산1동
25     연산9동
26     남천1동
27     망미2동
28      괘법동
29      괘법동
30      엄궁동
31      일광면
32      정관읍
33      중앙동
34     범일2동
35     부전2동
36     온천2동
37      좌1동
38      우2동
39      우2동
40      중앙동
41     부전2동
42     용호2동
43     대저2동
44     대저2동
45     대저2동
46      장안읍
47      장안읍
48      기장읍
49     범천1동
50     전포1동
51     온천1동
52     용호1동
53     전포1동
54     문현2동
55     덕천2동
56     덕천2동
57      좌1동
58      중1동
59     괴정3동
60     장림2동
61     장전2동
62     명지1동
63     거제1동
64      기장읍
65     부전2동
66     하단2동
67      괘법동
68      기장읍
69     괴정1동
70     온천2동
71     장림2동
72     다대2동
73     다대1동
74     부전2동
75     부암3동
76      수민동
77    청룡노포동
78      감전동
79     양정1동
80     부전2동
81     전포1동
82     당감2동
83  

In [44]:
gdf_joined = gdf_joined.groupby('d1').size().reset_index(name='대형마트_개수')

In [47]:
print(gdf_joined.to_string())

       d1  대형마트_개수
0    가야2동        1
1    감만2동        1
2     감전동        1
3    거제1동        1
4    거제2동        1
5     괘법동        3
6    괴정1동        1
7    괴정3동        2
8    구서1동        1
9     금곡동        1
10    기장읍        4
11    남산동        1
12   남천1동        1
13   다대1동        2
14   다대2동        1
15   당감2동        1
16   대저2동        3
17   덕천2동        2
18   망미2동        1
19    명륜동        1
20   명지1동        1
21   문현2동        3
22   반여1동        1
23   범일2동        1
24   범천1동        1
25   봉래1동        1
26   부암1동        2
27   부암3동        1
28   부전1동        1
29   부전2동        6
30    수민동        1
31   신평2동        1
32   양정1동        1
33    엄궁동        1
34   연산1동        1
35   연산2동        1
36   연산9동        1
37   온천1동        3
38   온천2동        3
39   용호1동        1
40   용호2동        1
41    우2동        3
42    우3동        2
43    일광면        1
44   장림1동        1
45   장림2동        3
46    장안읍        2
47   장전2동        1
48   전포1동        4
49    정관읍        2
50    좌1동        2
51    중1동   

In [48]:
gdf_joined = gdf_joined.rename(columns={'d1': '행정동'})

In [49]:
gdf_joined.head()

,행정동,대형마트_개수
0,가야2동,1
1,감만2동,1
2,감전동,1
3,거제1동,1
4,거제2동,1


In [50]:
gdf_joined.to_csv("행정동_매핑_마트데이터.csv", index=False, encoding='utf-8-sig')